# Modelo de Riesgo Crediticio — Red Neuronal (Probabilidad de Default)

Clasificación binaria de clientes **deudores vs. no deudores** con una red neuronal (TensorFlow/Keras).
El foco no es solo la precisión, sino elegir el **umbral de corte** que minimice la **pérdida esperada** del portafolio.

**Pipeline:** EDA → preprocesamiento → balanceo de clases (SMOTE) → red neuronal → evaluación (ROC-AUC, Precision-Recall) → umbral óptimo por costo.

> Coloca el archivo `Datos_Crediticios.csv` dentro de la carpeta `data/`. El dataset tiene 12 columnas; la variable objetivo es `Default` (1 = incumple).

## 1. Configuración e importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Carga y exploración de los datos (EDA)

In [ ]:
# El separador es ';' y la codificacion 'latin-1' (formato del archivo original)
df = pd.read_csv("data/Datos_Crediticios.csv", sep=";", encoding="latin-1")
print(f"Registros: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
# Tipos de dato y valores nulos por columna
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())

In [ ]:
# Distribucion de la variable objetivo (Default): revela el desbalance de clases
target = "Default"
dist = df[target].value_counts(normalize=True).rename({0: "No deudor (0)", 1: "Deudor (1)"})
print("Proporcion de clases:")
print((dist * 100).round(2).astype(str) + " %")

ax = df[target].value_counts().plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_title("Distribucion de la variable objetivo (Default)")
ax.set_xlabel("Clase"); ax.set_ylabel("Frecuencia")
plt.tight_layout(); plt.show()

In [ ]:
# Estadisticos descriptivos
df.describe().T

In [ ]:
# Matriz de correlacion: que variables se relacionan mas con el Default
plt.figure(figsize=(11, 8))
sns.heatmap(df.drop(columns=["ID"]).corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlacion")
plt.tight_layout(); plt.show()

## 3. Preprocesamiento

Se separa la variable objetivo, se descarta el identificador `ID`, se divide en entrenamiento/prueba de forma
**estratificada** (para conservar la proporcion de clases) y se estandarizan las variables. El escalador se **ajusta
solo con el train** para evitar fuga de informacion.

In [ ]:
FEATURES = [c for c in df.columns if c not in ["ID", target]]

X = df[FEATURES].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 4. Balanceo de clases con SMOTE

El desbalance hace que un modelo ingenuo prediga siempre "no deudor". SMOTE genera ejemplos sinteticos de la clase
minoritaria **solo en el conjunto de entrenamiento** (nunca en el de prueba).

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Antes de SMOTE:", np.bincount(y_train))
print("Despues de SMOTE:", np.bincount(y_train_bal))

## 5. Red neuronal (TensorFlow/Keras)

In [ ]:
def build_model(input_dim: int) -> tf.keras.Model:
    """Construye y compila la red neuronal para clasificacion binaria.

    Args:
        input_dim: Numero de variables de entrada.

    Returns:
        Modelo Keras compilado (perdida binary_crossentropy, metrica AUC).
    """
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(name="auc")])
    return model

model = build_model(X_train_bal.shape[1])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor="val_auc", mode="max",
                           patience=5, restore_best_weights=True)

history = model.fit(
    X_train_bal, y_train_bal,
    validation_split=0.2,
    epochs=50,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Perdida (binary_crossentropy)"); axes[0].legend()
axes[1].plot(history.history["auc"], label="train")
axes[1].plot(history.history["val_auc"], label="val")
axes[1].set_title("AUC"); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluacion del modelo

Se evalua sobre el conjunto de prueba con metricas apropiadas para clases desbalanceadas: **ROC-AUC** y
**Precision-Recall AUC**, ademas del reporte de clasificacion y la matriz de confusion.

In [ ]:
y_proba = model.predict(X_test).ravel()

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")

In [ ]:
# Curvas ROC y Precision-Recall
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec, rec, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("Curva ROC"); axes[0].legend()
axes[1].plot(rec, prec, label=f"PR (AUC = {pr_auc:.3f})", color="#C44E52")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Curva Precision-Recall"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Umbral de corte optimo (minimizar la perdida esperada)

Un umbral de 0.5 rara vez es optimo en negocio. Aprobar a un cliente que incumple (falso negativo) suele costar
**mucho mas** que rechazar a un buen cliente (falso positivo). Buscamos el umbral que minimiza la perdida esperada
segun una matriz de costos.

In [ ]:
def expected_loss(y_true, y_proba, threshold, cost_fn, cost_fp):
    """Perdida esperada para un umbral dado.

    Args:
        y_true: Etiquetas reales (0/1).
        y_proba: Probabilidades predichas de incumplimiento.
        threshold: Umbral de decision.
        cost_fn: Costo de un falso negativo (aprobar a un deudor).
        cost_fp: Costo de un falso positivo (rechazar a un buen cliente).

    Returns:
        Perdida total estimada.
    """
    y_pred = (y_proba >= threshold).astype(int)
    fn = np.sum((y_pred == 0) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return fn * cost_fn + fp * cost_fp

# Supuesto de negocio: un incumplimiento cuesta 5x lo que cuesta perder un buen cliente
COST_FN, COST_FP = 5.0, 1.0

thresholds = np.linspace(0.01, 0.99, 99)
losses = [expected_loss(y_test, y_proba, t, COST_FN, COST_FP) for t in thresholds]
best_threshold = thresholds[int(np.argmin(losses))]

print(f"Umbral optimo: {best_threshold:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(thresholds, losses)
plt.axvline(best_threshold, color="#C44E52", linestyle="--", label=f"optimo = {best_threshold:.2f}")
plt.xlabel("Umbral"); plt.ylabel("Perdida esperada"); plt.title("Perdida esperada vs. umbral")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Reporte final con el umbral optimo
y_pred = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred, target_names=["No deudor", "Deudor"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No deudor", "Deudor"]).plot(cmap="Blues")
plt.title(f"Matriz de confusion (umbral = {best_threshold:.2f})")
plt.tight_layout(); plt.show()

## 8. Conclusiones

- La red neuronal alcanza un **ROC-AUC** solido en la clasificacion de incumplimiento (ver salida de la celda de evaluacion).
- SMOTE permite que el modelo aprenda la clase minoritaria en lugar de ignorarla.
- El **umbral optimo** por costo traslada el modelo de una metrica estadistica a una **decision de negocio** (aprobar/rechazar) que minimiza la perdida esperada.

**Proximos pasos:** calibracion de probabilidades (Platt/isotonica), explicabilidad con SHAP y despliegue como API (FastAPI + Docker).